# EXP-2026-008 D2 on Colab GPU: the one-patient wiring test of the localized sampler

The card `experiments/registry/EXP-2026-008.md` holds the design and the frozen evaluation (revised after two reviews by Codex). The settings are in `configs/experiments/EXP-2026-008_d2.json`, and every step below calls the tested module `rhuh_synthesis/localized_wiring.py`.

- **Patient:** the accepted RHUH development case with the median pre-op core volume, chosen by that rule. A training patient: no validation or test patient is read.
- **Arms:** W0 to W7 with W1r, W4g and W6e, on a 12-slice slab, seeds 11, 22 and 33 with matched base noise. Also an empty-region check, a full-region check and an identical repeat.
- **Weights:** the step-1000 UNet of Stage 5 run r2, fetched from Drive by ID and checked against its SHA-256.
- **Inputs:** pre-op only, plus a supplied scenario. The real post-op scan is read only for descriptive, target-informed scores and for the review pages. A scenario is a counterfactual, not a forecast of the real outcome.

Runtime is not measured yet: roughly 20 to 40 minutes on a Colab GPU, model loading included. Colab asks for permission twice: Drive mount and the Drive API. Outputs (record, stacks, review pages) go to Colab's disk first. They are then uploaded through the Drive API to `RHUH-GBM/synthesis/EXP-2026-008/d2/<RUN_TAG>/`, and the last cell confirms every file by the server's MD5. The record and the pages are private: they hold a patient ID and patient images.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-d2"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
SETTINGS = "configs/experiments/EXP-2026-008_d2.json"
RUN_TAG = "d2_r3"  # a new tag for every new run (d2_r1 stopped, d2_r2 failed a check); a run folder with files is refused
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")  # from MyDrive; the API follows the shortcut
OUT_BASE, OUT_RUN = ("RHUH-GBM", "synthesis"), ("EXP-2026-008", "d2", RUN_TAG)

## 1. Drive, code from the fork, packages, run folders

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # before any CUDA work: repeatable cuBLAS
from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

from rhuh_synthesis.colab_packages import DATA, MODEL  # the one package list; it imports nothing

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DATA, *MODEL], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

settings = json.loads(Path(SETTINGS).read_text(encoding="utf-8"))
auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)
weights_run = DriveRun.open(api, DATA_ON_DRIVE, ("stage5_runs", settings["weights"]["stage5_run_tag"]),
                            "/content/d2_weights", expect_in_base=("split", "export_v1"))
LOCAL_RUN = Path(f"/content/d2_{RUN_TAG}")
assert not LOCAL_RUN.exists(), f"{LOCAL_RUN} exists: use a new RUN_TAG or a fresh runtime"
run_drive = DriveRun.open(api, OUT_BASE, OUT_RUN, LOCAL_RUN, expect_in_base=("EXP-2026-001",), create=True)
print(run_drive.preflight())  # a real write, update, read and removal on Drive, before any GPU time
STARTED = time.perf_counter()

## 2. Models: the Stage 5 r2 UNet (hash checked), the frozen VAE, CLIP and DDIM

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.model import reference_ddim_scheduler, scheduler_mismatches
from rhuh_synthesis.overfit import load_trained_unet
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import clip_encoder, load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
part1 = json.loads(weights_run.fetch(settings["weights"]["record"]).read_text(encoding="utf-8"))
WEIGHTS = weights_run.fetch(part1["weights"]["file"])  # by ID, checked by MD5
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, hf_hub_download(MSF_REPO, MSF_FILE))
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
unet11, weights_report = load_trained_unet(unet4, WEIGHTS, device="cuda")
del unet4
torch.cuda.empty_cache()
assert weights_report["sha256"] == part1["weights"]["sha256"], "The weights on Drive differ from Stage 5 r2: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
ddim = reference_ddim_scheduler()
assert scheduler_mismatches(ddim) == []
head = {
    "commit": commit, "code_version": code_version(WORK), "settings_sha256": config_sha256(settings),
    "exp001_config_sha256": config_sha256(config), "environment": environment(), "torch": torch.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0), "cpu_count": os.cpu_count(),
    "weights": weights_report, "weights_run": settings["weights"]["stage5_run_tag"],
}
print({k: head[k] for k in ("commit", "torch", "cuda", "gpu")}, "| weights", weights_report["sha256"][:12])

## 3. The patient, chosen by the frozen rule

In [ ]:
from rhuh_synthesis.localized_wiring import choose_patient, core_volume_ml
from rhuh_synthesis.pilot import local_copies

PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
assert policy.get("status") == "FROZEN", "The acceptance policy is not frozen"
train = set(split["roles"]["train"]["patients"])
candidates = [p for p in split["development_cases_pinned_to_train"] if p in train and p not in policy["excluded_training"]]
paths = local_copies({p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in candidates}, "/content/d2_exports")
core_ml = {}
for candidate, path in paths.items():
    with np.load(path) as data:
        core_ml[candidate] = core_volume_ml({"pre_label": data["pre_label"], "frame_affine": data["frame_affine"]})
patient, choice = choose_patient(split, policy, core_ml)
assert patient in train and patient not in set(split["roles"]["val"]["patients"]) | set(split["roles"]["test"]["patients"])
with np.load(paths[patient]) as data:
    arrays = {key: data[key] for key in data.files}
print("patient choice (counts only):", choice)

## 4. Every arm and check, the readouts and the write-once record

In [ ]:
from rhuh_synthesis.localized_wiring import run_wiring, summary
from rhuh_synthesis.manifest import write_once_json

say = lambda message: print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)
record = run_wiring(
    arrays, settings, config["evaluation"], config["text"], unet=unet11, vae=vae, scheduler=ddim,
    encode_text=clip_encoder(tokenizer, text_encoder, "cuda"), codec=config["representation"]["mask_codec"],
    crop_mm=config["representation"]["crop_mm"], out_px=config["representation"]["out_px"],
    num_inference_steps=config["sampling"]["num_inference_steps"], guidance_scale=config["sampling"]["guidance_scale"],
    out_dir=LOCAL_RUN, device="cuda", progress=say, grid_options={"panel_inches": 4.0, "dpi": 96},
)
record.update(head, patient=patient, patient_choice=choice)  # private record: it names the patient
path, digest = write_once_json(LOCAL_RUN / "d2_record.json", record)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")

## 5. Confirm the upload and print the summary

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: " + "/".join(OUT_BASE + OUT_RUN))
print("SUMMARY")
print(json.dumps(summary(record), indent=1))

## What to send back

Copy everything printed after `SUMMARY`. It holds the decision, the checks and per-arm numbers, but no patient ID. If a cell stops with an error, send the error message. An error that names an arm, a seed and a slice means a non-finite decoded image: the run stopped on purpose. The review pages in `grids/` on Drive show patient images: keep them private.